# PREFFECT: a worked example

The same steps as the documentation's [worked example](../docs_src/vignette.md), as a notebook: write a small simulated cohort, check it, fit a model, and apply it to data the model has never seen. About two minutes end to end.

Each cell calls the command line from Python, so what runs here is exactly what `preffect check`, `preffect train` and `preffect apply` run. Install PREFFECT first (see the README), then run the cells in order. Everything is written into the folder the notebook runs in.

In [ ]:
from preffect.cli import main

# A small simulated cohort: three splits with no sample in common, for each of the three models.
assert main(["example-data", "--out", "vignette_data"]) == 0

## 1. Check the data first

`check` reads the counts without fitting anything. The last line it prints says whether any training sample reappears in the validation set, which is how the problem with version 0.1's vignettes would have been caught.

In [ ]:
assert main(["check", "--train", "vignette_data/simple/train",
             "--validation", "vignette_data/simple/validation"]) == 0

## 2. Fit

`--patience 5` stops once the validation loss has not fallen for five epochs, and the model written is the best epoch's. `--seed` makes the run repeatable; `--threads 4` suits a small example.

In [ ]:
assert main(["train", "--train", "vignette_data/simple/train",
             "--validation", "vignette_data/simple/validation",
             "--out", "model.pt",
             "--epochs", "80", "--batch-size", "40", "--seed", "0", "--threads", "4",
             "--patience", "5"]) == 0

## 3. Apply it to data it has never seen

The test split was never seen during fitting and never used to decide when to stop, so what comes out is a genuine held-out result.

In [ ]:
assert main(["apply", "--model", "model.pt",
             "--data", "vignette_data/simple/test",
             "--out", "cleaned.h5ad", "--threads", "4"]) == 0

## 4. What came out

`cleaned.h5ad` carries the adjusted counts in `X`, the predicted proportions in `layers['composition']`, and the latent summary of each sample in `obsm['latent']`.

In [ ]:
import anndata as ad

cleaned = ad.read_h5ad("cleaned.h5ad")
print(cleaned)
print("latent summary:", cleaned.obsm["latent"].shape)

## Next

- The [graph model](../docs_src/vignette.md#the-graph-model) on `vignette_data/single/`, which carries a sample-sample network with real edges.
- [Using it](../docs_src/guide.md) for imputation, clustering and figures from Python.
- [The command line](../docs_src/command-line.md) for every option.